# Test Bedrock Models

In [ ]:
import boto3

client = boto3.client("bedrock", region_name="ap-southeast-2")
models = client.list_foundation_models(byProvider="Anthropic")

for m in models["modelSummaries"]:
    print(m["modelId"], "|", m["modelName"], "|", m.get("inferenceTypesSupported"))

In [ ]:
import boto3

client = boto3.client("bedrock", region_name="ap-southeast-2")
response = client.get_foundation_model(
    modelIdentifier="anthropic.claude-sonnet-4-20250514-v1:0"
)
print(response)

In [ ]:
import boto3
import json

client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

# All INFERENCE_PROFILE models from your list_foundation_models output
# Testing multiple AU prefix formats per model
candidates = [
    # Claude Sonnet 4.5
    "ap-southeast-2.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "au.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "anthropic.claude-sonnet-4-5-20250929-v1:0",
    # Claude Haiku 4.5
    "ap-southeast-2.anthropic.claude-haiku-4-5-20251001-v1:0",
    "au.anthropic.claude-haiku-4-5-20251001-v1:0",
    "anthropic.claude-haiku-4-5-20251001-v1:0",
    # Claude Sonnet 4.6
    "ap-southeast-2.anthropic.claude-sonnet-4-6",
    "au.anthropic.claude-sonnet-4-6",
    "anthropic.claude-sonnet-4-6",
    # Claude Opus 4.5
    "ap-southeast-2.anthropic.claude-opus-4-5-20251101-v1:0",
    "au.anthropic.claude-opus-4-5-20251101-v1:0",
    "anthropic.claude-opus-4-5-20251101-v1:0",
]

body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 50,
    "messages": [{"role": "user", "content": "Say hello"}]
})

print(f"{'Model ID':<65} {'Result'}")
print("-" * 85)

for model_id in candidates:
    try:
        response = client.invoke_model(
            modelId=model_id,
            contentType="application/json",
            accept="application/json",
            body=body
        )
        result = json.loads(response["body"].read())
        print(f"{model_id:<65} SUCCESS - {result['content'][0]['text'][:30]}")
    except Exception as e:
        code = e.response["Error"]["Code"] if hasattr(e, "response") else type(e).__name__
        print(f"{model_id:<65} {code}")

In [ ]:
client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

legacy_models = [
    "anthropic.claude-3-sonnet-20240229-v1:0",
    "anthropic.claude-3-haiku-20240307-v1:0",
    "anthropic.claude-3-5-sonnet-20241022-v2:0",
]

body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 50,
    "messages": [{"role": "user", "content": "Say hello"}]
})

for model_id in legacy_models:
    try:
        response = client.invoke_model(
            modelId=model_id,
            contentType="application/json",
            accept="application/json",
            body=body
        )
        result = json.loads(response["body"].read())
        print(f"SUCCESS: {model_id} -> {result['content'][0]['text'][:50]}")
    except Exception as e:
        print(f"FAILED: {model_id} -> {e.response['Error']['Code']}: {e.response['Error']['Message'][:80]}")

In [ ]:
import boto3
import json

client = boto3.client("bedrock-runtime", region_name="ap-southeast-2")

models = [
    ("Amazon Nova Micro",  "amazon.nova-micro-v1:0",  "nova"),
    ("Amazon Nova Lite",   "amazon.nova-lite-v1:0",   "nova"),
    ("Amazon Nova Pro",    "amazon.nova-pro-v1:0",    "nova"),
]

for name, model_id, provider in models:
    try:
        if provider == "nova":
            body = json.dumps({
                "messages": [{"role": "user", "content": [{"text": "Say hello"}]}]
            })
        
        response = client.invoke_model(
            modelId=model_id,
            contentType="application/json",
            accept="application/json",
            body=body
        )
        result = json.loads(response["body"].read())
        text = result["output"]["message"]["content"][0]["text"]
        print(f"SUCCESS: {name} -> {text[:60]}")
    except Exception as e:
        print(f"FAILED:  {name} -> {e.response['Error']['Code']}")